# Student Lifestyle & GPA Classification
Predicting GPA category (Low / Medium / High) from student lifestyle factors using Logistic Regression, Decision Tree, and Random Forest classifiers.

## Task 1: Inspect the Dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('student_lifestyle_dataset.csv')
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'student_lifestyle_dataset.csv'

In [ ]:
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nTotal missing values:", df.isnull().sum().sum())

In [ ]:
df.info()
df.describe()

## Task 2: Prepare the Dataset
No missing values were found in this dataset. We still demonstrate a missing-value handling step for completeness, then create our classification target and encode categorical features.

In [ ]:
# Handle missing values (none present, but included for completeness)
df = df.dropna()
print("Shape after dropna:", df.shape)

In [ ]:
# Create classification target: bin GPA into 3 balanced categories
df['GPA_Category'] = pd.qcut(df['GPA'], q=3, labels=['Low', 'Medium', 'High'])
print(df['GPA_Category'].value_counts())

plt.figure(figsize=(5,4))
sns.countplot(x='GPA_Category', data=df, order=['Low','Medium','High'])
plt.title('Distribution of GPA Category')
plt.savefig('gpa_category_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode categorical feature: Stress_Level
le_stress = LabelEncoder()
df['Stress_Level_encoded'] = le_stress.fit_transform(df['Stress_Level'])
print("Stress_Level classes:", dict(zip(le_stress.classes_, le_stress.transform(le_stress.classes_))))

# Encode target
le_target = LabelEncoder()
df['GPA_Category_encoded'] = le_target.fit_transform(df['GPA_Category'])
print("GPA_Category classes:", dict(zip(le_target.classes_, le_target.transform(le_target.classes_))))

In [ ]:
plt.figure(figsize=(6,5))
corr_cols = ['Study_Hours_Per_Day','Extracurricular_Hours_Per_Day','Sleep_Hours_Per_Day',
             'Social_Hours_Per_Day','Physical_Activity_Hours_Per_Day','GPA']
sns.heatmap(df[corr_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

## Task 3: Build Classification Models
We use Study_Hours, Extracurricular_Hours, Sleep_Hours, Social_Hours, Physical_Activity_Hours, and Stress_Level (encoded) as features to predict GPA_Category.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

feature_cols = ['Study_Hours_Per_Day','Extracurricular_Hours_Per_Day','Sleep_Hours_Per_Day',
                 'Social_Hours_Per_Day','Physical_Activity_Hours_Per_Day','Stress_Level_encoded']

X = df[feature_cols]
y = df['GPA_Category_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Train shape:", X_train.shape, " Test shape:", X_test.shape)

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200)
}

trained_models = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    trained_models[name] = model
    predictions[name] = preds
    print(f"{name} trained.")

## Task 4: Evaluate the Models

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, f1_score, confusion_matrix

results = []
for name, preds in predictions.items():
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds, average='weighted')
    f1 = f1_score(y_test, preds, average='weighted')
    results.append({'Model': name, 'Accuracy': round(acc,4), 'Precision': round(prec,4), 'F1-score': round(f1,4)})

results_df = pd.DataFrame(results)
results_df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
labels = le_target.classes_

for ax, (name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels, ax=ax)
    ax.set_title(name)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
results_df.set_index('Model')[['Accuracy','Precision','F1-score']].plot(kind='bar')
plt.title('Model Comparison')
plt.ylabel('Score')
plt.ylim(0,1)
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary
- The dataset has 2000 rows and 8 columns with no missing values.
- GPA was binned into three balanced classes (Low / Medium / High) using quantiles to enable classification.
- `Stress_Level` was label-encoded; numerical lifestyle features were used as-is.
- Three classifiers (Logistic Regression, Decision Tree, Random Forest) were trained and evaluated.
- See the results table and confusion matrices above for accuracy, precision, F1-score, and per-class performance comparisons.